### Final comparison on ASVspoof2019-LA eval

The goal of this final step is to compare the two final models found in our respective step 2s:

- Optimized SVM (Nora pipeline)

- Optimized Random Forest (Maryam pipeline)

The objective is to verify robustness under real-world conditions on unlabeled spoofed audio.

A better ROC-AUC and a high F1 score indicate a better ability to differentiate between bona fide and spoofed audio.

We will compare the metrics to determine which of the two pipelines generalizes best to the final test set.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import joblib

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

SRC_DIR = Path("./src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_utils import load_asvspoof_protocol, attach_paths_by_utt

print("Utils loaded.")


Utils loaded.


In [2]:
model_svm = joblib.load(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\data\model_svm.pkl")
model_rf  = joblib.load(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\data\model_rf.pkl")

print("Loaded models :")
print(" - SVM :", model_svm)
print(" - Random Forest :", model_rf)


Loaded models :
 - SVM : Pipeline(steps=[('standardscaler', StandardScaler()),
                ('svc',
                 SVC(C=1, class_weight='balanced', probability=True,
                     random_state=42))])
 - Random Forest : RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)


C:\Users\norab\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\norab\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [3]:
DATA_ROOT = Path(r"C:\Users\norab\Documents\Cours_ESILV_A4\Machine Learning\deepfake-audio-project\archive\LA")

PROTO_EVAL = DATA_ROOT / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.eval.trl.txt"
AUDIO_ROOT_EVAL = DATA_ROOT / "ASVspoof2019_LA_eval" / "flac"


In [4]:
eval_df = attach_paths_by_utt(load_asvspoof_protocol(PROTO_EVAL), AUDIO_ROOT_EVAL)

print("Eval size:", eval_df.shape)
eval_df.head()


Eval size: (71237, 3)


,utt,is_spoof,path
0,LA_E_2834763,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
1,LA_E_8877452,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
2,LA_E_6828287,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
3,LA_E_6977360,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...
4,LA_E_5932896,1,C:\Users\norab\Documents\Cours_ESILV_A4\Machin...


In [5]:
def extract_features(path, sr=16000, n_mfcc=13):
    y, _ = librosa.load(path, sr=sr)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

# We are testing on 5000 randomly selected samples because all the evaluation files would be too numerous.
eval_sample = eval_df.sample(5000, random_state=42)

X_eval = np.vstack([extract_features(p) for p in eval_sample["path"]])
y_eval = eval_sample["is_spoof"].to_numpy()

X_eval.shape, y_eval.shape


((5000, 26), (5000,))

In [6]:
def evaluate(name, model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y, y_pred))
    print("F1-score :", f1_score(y, y_pred))
    print("ROC-AUC  :", roc_auc_score(y, y_proba))
    print("Confusion matrix:\n", confusion_matrix(y, y_pred))

evaluate("SVM (final)", model_svm, X_eval, y_eval)
evaluate("Random Forest (final)", model_rf, X_eval, y_eval)



===== SVM (final) =====
Accuracy : 0.8798
F1-score : 0.9328566640598815
ROC-AUC  : 0.886613059811743
Confusion matrix:
 [[ 224  306]
 [ 295 4175]]

===== Random Forest (final) =====
Accuracy : 0.884
F1-score : 0.9370386452453322
ROC-AUC  : 0.8658399814275464
Confusion matrix:
 [[ 104  426]
 [ 154 4316]]


### **Analysis of results — SVM**

- Accuracy : 0.8798  
- F1-score : 0.9329  
- ROC-AUC : 0.8866

Confusion Matrix :

              Predicted bonafide   Predicted spoof 
    True bonafide    ((224)            (306)
    True spoof        (295)            (4175))

**Interpretation :**
The SVM maintains very good overall performance. It correctly detects almost all spoof attacks (4175 out of 4470), which explains its high F1 score.
However, it remains more susceptible to false positives than Random Forest, resulting in 306 misclassified bonafide attacks.
This behavior is consistent with the initially unbalanced structure of the LA data.

---

### **Analysis of results — Random Forest**

- Accuracy : 0.8840  
- F1-score : 0.9370  
- ROC-AUC : 0.8658

Confusion Matrix :

              Predicted bonafide   Predicted spoof 
    True bonafide    ((104)            (426)
    True spoof        (154)            (4316))

**Interpretation :**  
Random Forest achieves the best overall performance, notably due to a significant reduction in false positives (only 104 bonafide distributions classified as spoof).
Its F1 score is the best of the two models, making it a more robust option for real-world use.
Its ROC-AUC is slightly lower than that of SVM, indicating a slightly less clear separation between the spoof and bonafide distributions, but this does not affect its practical effectiveness.

---

### **General conclusion of Step 3**

Both models demonstrate excellent generalization capabilities on entirely new data:
Random Forest appears to be the most operationally robust model (better accuracy, better F1 score, fewer errors on bonafide).
SVM remains very competitive and displays the best ROC-AUC, indicating good theoretical separation between classes.

These results demonstrate that our MFCC-based (26 dimensions) approach, combined with the models covered in the course, can provide a solid foundation for audio deepfake detection.
